In [1]:
# ===============================
# 1. IMPORT LIBRARIES
# ===============================

from pathlib import Path
import pandas as pd
import numpy as np
import re

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

In [2]:
# ===============================
# 2. LOAD DATASET
# ===============================

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "dataset" / "naukri_dataset.csv"

df = pd.read_csv(DATA_PATH)

In [3]:
print("Shape of dataset:", df.shape)
df.head()

Shape of dataset: (30000, 11)


,Uniq Id,Crawl Timestamp,Job Title,Job Salary,Job Experience Required,Key Skills,Role Category,Location,Functional Area,Industry,Role
0,9be62c49a0b7ebe982a4af1edaa7bc5f,2019-07-05 01:46:07 +0000,Digital Media Planner,Not Disclosed by Recruiter,5 - 10 yrs,Media Planning| Digital Media,Advertising,Mumbai,"Marketing , Advertising , MR , PR , Media Planning","Advertising, PR, MR, Event Management",Media Planning Executive/Manager
1,3c52d436e39f596b22519da2612f6a56,2019-07-06 08:04:50 +0000,Online Bidding Executive,Not Disclosed by Recruiter,2 - 5 yrs,pre sales| closing| software knowledge| clients| requirements| negotiating| client| online bidding| good communication| technology,Retail Sales,"Pune,Pune","Sales , Retail , Business Development","IT-Software, Software Services",Sales Executive/Officer
2,ffad8a2396c60be2bf6d0e2ff47c58d4,2019-08-05 15:50:44 +0000,Trainee Research/ Research Executive- Hi- Tech Operations,Not Disclosed by Recruiter,0 - 1 yrs,Computer science| Fabrication| Quality check| Intellectual property| Electronics| Support services| Research| Management| Human resource management| Research Executive,R&D,Gurgaon,"Engineering Design , R&D","Recruitment, Staffing",R&D Executive
3,7b921f51b5c2fb862b4a5f7a54c37f75,2019-08-05 15:31:56 +0000,Technical Support,"2,00,000 - 4,00,000 PA.",0 - 5 yrs,Technical Support,Admin/Maintenance/Security/Datawarehousing,Mumbai,"IT Software - Application Programming , Maintenance","IT-Software, Software Services",Technical Support Engineer
4,2d8b7d44e138a54d5dc841163138de50,2019-07-05 02:48:29 +0000,Software Test Engineer -hyderabad,Not Disclosed by Recruiter,2 - 5 yrs,manual testing| test engineering| test cases| web testing| web technologies,Programming & Design,Hyderabad,IT Software - QA & Testing,"IT-Software, Software Services",Testing Engineer


In [4]:
# ===============================
# 3. CLEAN COLUMN NAMES
# ===============================

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print("Updated columns:\n", df.columns.tolist())

Updated columns:
 ['uniq_id', 'crawl_timestamp', 'job_title', 'job_salary', 'job_experience_required', 'key_skills', 'role_category', 'location', 'functional_area', 'industry', 'role']


In [5]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   uniq_id                  30000 non-null  object
 1   crawl_timestamp          30000 non-null  object
 2   job_title                29425 non-null  object
 3   job_salary               29950 non-null  object
 4   job_experience_required  29427 non-null  object
 5   key_skills               28729 non-null  object
 6   role_category            27695 non-null  object
 7   location                 29423 non-null  object
 8   functional_area          29427 non-null  object
 9   industry                 29427 non-null  object
 10  role                     29099 non-null  object
dtypes: object(11)
memory usage: 2.5+ MB


In [7]:
# Convert timestamp column to datetime
df["crawl_timestamp"] = pd.to_datetime(df["crawl_timestamp"], errors="coerce")
df.dtypes

uniq_id                                 object
crawl_timestamp            datetime64[ns, UTC]
job_title                               object
job_salary                              object
job_experience_required                 object
key_skills                              object
role_category                           object
location                                object
functional_area                         object
industry                                object
role                                    object
dtype: object

In [8]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_df = pd.DataFrame({
    "Missing Count": missing,
    "Missing %": (missing / len(df)) * 100
})

missing_df

,Missing Count,Missing %
role_category,2305,7.683333
key_skills,1271,4.236667
role,901,3.003333
location,577,1.923333
job_title,575,1.916667
functional_area,573,1.910000
job_experience_required,573,1.910000
industry,573,1.910000
job_salary,50,0.166667
crawl_timestamp,0,0.000000


### Salary Cleaning

In [9]:
df["job_salary"] = df["job_salary"].str.strip()

# Salary disclosed flag
df["salary_disclosed"] = np.where(
    df["job_salary"].str.contains("not disclosed", case=False, na=False),
    0,
    1
)

In [10]:
def extract_salary(salary):
    if pd.isna(salary):
        return np.nan, np.nan
    
    salary = salary.lower().strip()
    
    if "not disclosed" in salary:
        return np.nan, np.nan
    
    salary = salary.replace(",", "")
    
    numbers = re.findall(r'\d+', salary)
    
    # If only one number (e.g., "3 LPA")
    if len(numbers) == 1:
        value = int(numbers[0])
        if "lakh" in salary or "lac" in salary:
            value *= 100000
        if value < 10000:
            return np.nan, np.nan
        return value, value
    
    # If range format
    if len(numbers) >= 2:
        min_sal = int(numbers[0])
        max_sal = int(numbers[1])
        
        if "lakh" in salary or "lac" in salary:
            min_sal *= 100000
            max_sal *= 100000
        
        if min_sal < 10000 or max_sal < 10000:
            return np.nan, np.nan
        
        return min_sal, max_sal
    
    return np.nan, np.nan

df[["salary_min", "salary_max"]] = df["job_salary"].apply(
    lambda x: pd.Series(extract_salary(x))
)

df["salary_avg"] = (df["salary_min"] + df["salary_max"]) / 2

### Remove Extreme Salary Outliers

In [11]:
# Remove extreme outliers using 99th percentile
upper_cap = df["salary_avg"].quantile(0.99)

df.loc[
    df["salary_avg"] > upper_cap,
    ["salary_min", "salary_max", "salary_avg"]
] = np.nan

df["salary_avg"].describe()

count    7.959000e+03
mean     5.684929e+05
std      5.149548e+05
min      1.000000e+04
25%      2.500000e+05
50%      3.750000e+05
75%      6.750000e+05
max      3.500000e+06
Name: salary_avg, dtype: float64

### Experience Cleaning

In [12]:
# ===============================
# 7. EXPERIENCE PROCESSING
# ===============================

def extract_experience(exp):
    if pd.isna(exp):
        return np.nan, np.nan
    
    exp = exp.lower().strip()
    numbers = re.findall(r'\d+', exp)
    
    if len(numbers) < 2:
        return np.nan, np.nan
    
    min_exp = int(numbers[0])
    max_exp = int(numbers[1])
    
    if min_exp > 40 or max_exp > 50:
        return np.nan, np.nan
    
    return min_exp, max_exp

df[["exp_min", "exp_max"]] = df["job_experience_required"].apply(
    lambda x: pd.Series(extract_experience(x))
)

df["exp_avg"] = (df["exp_min"] + df["exp_max"]) / 2

df["exp_avg"].describe()

count    29231.000000
mean         5.119189
std          3.491235
min          0.000000
25%          3.000000
50%          4.500000
75%          6.500000
max         30.000000
Name: exp_avg, dtype: float64

In [13]:
# ===============================
# 8. LOCATION CLEANING
# ===============================

def clean_location(loc):
    if pd.isna(loc):
        return np.nan
    
    loc = loc.title().strip()
    
    cities = [c.strip() for c in loc.split(",")]
    
    mapping = {
        "Bangalore": "Bengaluru",
        "Delhi Ncr": "Delhi"
    }
    
    primary_city = cities[0]
    primary_city = mapping.get(primary_city, primary_city)
    
    return primary_city

df["location_cleaned"] = df["location"].apply(clean_location)

df["location_cleaned"].value_counts().head(10)

location_cleaned
Bengaluru    6039
Mumbai       4339
Delhi        3127
Pune         2687
Hyderabad    2604
Gurgaon      2140
Chennai      1947
Kolkata      1456
Noida        1333
Ahmedabad    1234
Name: count, dtype: int64

In [14]:
# ===============================
# 9. SKILL CLEANING
# ===============================

def clean_skills(skills):
    if pd.isna(skills):
        return np.nan
    
    skills = skills.lower()
    skills = skills.replace("|", ",")
    skills = skills.replace("/", ",")
    
    skills = re.sub(r"[^a-z0-9, ]", "", skills)
    
    skill_list = [s.strip() for s in skills.split(",") if s.strip() != ""]
    
    # Remove duplicates while preserving order
    skill_list = list(dict.fromkeys(skill_list))
    
    return ", ".join(skill_list)

df["key_skills_cleaned"] = df["key_skills"].apply(clean_skills)

In [15]:
# ===============================
# 10. UNIQUE VALUE COUNTS
# ===============================

for col in df.columns:
    print(f"{col} : {df[col].nunique()}")

uniq_id : 30000
crawl_timestamp : 28865
job_title : 23884
job_salary : 1167
job_experience_required : 256
key_skills : 26909
role_category : 206
location : 2573
functional_area : 72
industry : 122
role : 649
salary_disclosed : 2
salary_min : 65
salary_max : 92
salary_avg : 169
exp_min : 24
exp_max : 33
exp_avg : 54
location_cleaned : 653
key_skills_cleaned : 26723


In [16]:
df.describe()

,salary_disclosed,salary_min,salary_max,salary_avg,exp_min,exp_max,exp_avg
count,30000.000000,7.959000e+03,7.959000e+03,7.959000e+03,29231.000000,29231.000000,29231.000000
mean,0.307967,4.150110e+05,7.219747e+05,5.684929e+05,3.355205,6.883172,5.119189
std,0.461660,4.167405e+05,6.317259e+05,5.149548e+05,3.123892,3.988360,3.491235
min,0.000000,1.000000e+04,1.000000e+04,1.000000e+04,0.000000,0.000000,0.000000
25%,0.000000,1.750000e+05,3.000000e+05,2.500000e+05,1.000000,5.000000,3.000000
50%,0.000000,2.500000e+05,5.000000e+05,3.750000e+05,2.000000,6.000000,4.500000
75%,1.000000,5.000000e+05,9.000000e+05,6.750000e+05,5.000000,9.000000,6.500000
max,1.000000,3.000000e+06,5.000000e+06,3.500000e+06,25.000000,35.000000,30.000000


In [17]:
# ===============================
# 12. SAVE CLEANED DATA
# ===============================

processed_dir = PROJECT_ROOT / "dataset"
processed_dir.mkdir(parents=True, exist_ok=True)

df.to_csv(processed_dir / "naukri_cleaned.csv", index=False)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.
